# Eval 3.1 - Metric Expansion on the Revised Branch

This notebook extends `eval-3` without changing the revised undirected structural branch itself.
It adds:

- ranking metrics at `@10`
- `ROC-AUC`
- `PR-AUC`

The model family, split, and cached similarity matrices come from the existing `eval-3` branch.


In [1]:
import json
import math
import pickle
import sys
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "WikiCS" / "custom-wiki").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CUSTOM_WIKI_DIR = REPO_ROOT / "WikiCS" / "custom-wiki"
DATA_PATH = CUSTOM_WIKI_DIR / "data" / "data-embeddings.json"
BGE_PATH = CUSTOM_WIKI_DIR / "cap-embeddings" / "BAAI_bge-m3" / "master_embeddings.parquet"
SPLIT_PATH = CUSTOM_WIKI_DIR / "cache" / "node2vec-2" / "undirected_split.pkl"
EVAL3_CACHE_DIR = CUSTOM_WIKI_DIR / "cache" / "eval-3"
CACHE_DIR = CUSTOM_WIKI_DIR / "cache" / "eval-3.1"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CUSTOM_WIKI_DIR))
from utils.graph_utils import load_graph_data  # noqa: E402

SEED = 42
TOP_K = 10
K_RRF = 60.0
print("Repo root:", REPO_ROOT)
print("Eval-3 cache:", EVAL3_CACHE_DIR)
print("Eval-3.1 cache:", CACHE_DIR)


Repo root: C:\programming\github-repos\graph-ending
Eval-3 cache: C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3
Eval-3.1 cache: C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.1


## Step 1 - Load the revised branch and eval-3 artifacts

We reuse the same undirected split and the same cached similarity matrices already produced for `eval-3`.


In [2]:
nodes, id_to_title, title_to_id, graph = load_graph_data(str(DATA_PATH))
graph_undirected = graph.to_undirected()

with open(SPLIT_PATH, "rb") as f:
    split = pickle.load(f)

E_test = [tuple(map(int, edge)) for edge in split["E_test"]]
eval_pairs = [(u, v) for u, v in E_test] + [(v, u) for u, v in E_test]
eval_nodes = set(split["eval_nodes"])

df_ids = pd.read_parquet(BGE_PATH, columns=["id"]).sort_values("id").reset_index(drop=True)
full_ids = df_ids["id"].astype(int).to_numpy()
full_id_to_idx = {int(node_id): idx for idx, node_id in enumerate(full_ids)}

with open(EVAL3_CACHE_DIR / "gcn_only_results_eval3.pkl", "rb") as f:
    gcn_data = pickle.load(f)
with open(EVAL3_CACHE_DIR / "graphsage_results_eval3.pkl", "rb") as f:
    sage_data = pickle.load(f)

gcn_id_to_idx = {int(k): int(v) for k, v in gcn_data["node_to_idx"].items()}
sage_id_to_idx = {int(k): int(v) for k, v in sage_data["node_to_idx"].items()}

common_ids_rrf = sorted(set(full_id_to_idx.keys()) & set(gcn_id_to_idx.keys()))
bge_common_indices_rrf = np.array([full_id_to_idx[node_id] for node_id in common_ids_rrf], dtype=np.int32)
gcn_common_indices_rrf = np.array([gcn_id_to_idx[node_id] for node_id in common_ids_rrf], dtype=np.int32)
common_pos_rrf = {node_id: idx for idx, node_id in enumerate(common_ids_rrf)}

print("Nodes:", len(nodes))
print("Undirected test edges:", len(E_test))
print("Directed evaluation pairs:", len(eval_pairs))
print("Eval nodes:", len(eval_nodes))
print("Embedding ids:", len(full_ids))
print("GCN ids:", len(gcn_id_to_idx))
print("GraphSAGE ids:", len(sage_id_to_idx))


Nodes: 11701
Undirected test edges: 43223
Directed evaluation pairs: 86446
Eval nodes: 9022
Embedding ids: 11701
GCN ids: 11701
GraphSAGE ids: 11701


## Step 2 - Load eval-3 similarity matrices

Important detail: the direct similarity matrices do not all share the same row/column id lookup.
BGE and Node2Vec-family matrices follow the sorted BGE id order, while GCN and GraphSAGE use the
`node_to_idx` mappings from their revised-branch caches.


In [3]:
def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


sim_bge = load_pickle(EVAL3_CACHE_DIR / "sim_bge.pkl")
sim_n2v_uncorr = load_pickle(EVAL3_CACHE_DIR / "sim_n2v_undirected_uncorrected.pkl")
sim_n2v_corr = load_pickle(EVAL3_CACHE_DIR / "sim_n2v_undirected_corrected.pkl")
sim_comb_uncorr = load_pickle(EVAL3_CACHE_DIR / "sim_combined_undirected_uncorrected.pkl")
sim_comb_corr = load_pickle(EVAL3_CACHE_DIR / "sim_combined_undirected_corrected.pkl")
sim_gcn = load_pickle(EVAL3_CACHE_DIR / "sim_gcn.pkl")
sim_sage = load_pickle(EVAL3_CACHE_DIR / "sim_sage.pkl")
sim_pool_uncorr = load_pickle(EVAL3_CACHE_DIR / "sim_pooled_combined_uncorrected.pkl")
sim_pool_corr = load_pickle(EVAL3_CACHE_DIR / "sim_pooled_combined_corrected.pkl")

direct_models = [
    {"name": "1. BGE-M3 Only", "sim": sim_bge, "id_to_idx": full_id_to_idx},
    {"name": "2. Node2Vec (uncorrected)", "sim": sim_n2v_uncorr, "id_to_idx": full_id_to_idx},
    {"name": "3. Node2Vec (corrected)", "sim": sim_n2v_corr, "id_to_idx": full_id_to_idx},
    {"name": "4. BGE-M3 + Node2Vec (uncorrected)", "sim": sim_comb_uncorr, "id_to_idx": full_id_to_idx},
    {"name": "5. BGE-M3 + Node2Vec (corrected)", "sim": sim_comb_corr, "id_to_idx": full_id_to_idx},
    {"name": "6. GCN Only", "sim": sim_gcn, "id_to_idx": gcn_id_to_idx},
    {"name": "8. GraphSAGE Only", "sim": sim_sage, "id_to_idx": sage_id_to_idx},
    {"name": "9. BGE pooled + Node2Vec (uncorrected)", "sim": sim_pool_uncorr, "id_to_idx": full_id_to_idx},
    {"name": "10. BGE pooled + Node2Vec (corrected)", "sim": sim_pool_corr, "id_to_idx": full_id_to_idx},
]

print("Direct models ready:", len(direct_models))
print("BGE matrix shape:", sim_bge.shape)
print("GCN matrix shape:", sim_gcn.shape)


Direct models ready: 9
BGE matrix shape: (11701, 11701)
GCN matrix shape: (11701, 11701)


## Step 3 - Ranking helpers at cutoff 10

This branch still uses the same single-positive evaluation setup as `eval-3`.
Because each query has exactly one relevant target, `Recall@10` is numerically identical to `Hit@10`,
but we still report it explicitly for consistency with the requested metric family.


In [4]:
def safe_row_scores(sim_matrix: np.ndarray, source_idx: int) -> np.ndarray:
    scores = np.asarray(sim_matrix[source_idx], dtype=np.float32).copy()
    if source_idx < len(scores):
        scores[source_idx] = -np.inf
    finite_mask = np.isfinite(scores)
    if finite_mask.any() and (~finite_mask).any():
        floor = float(np.min(scores[finite_mask])) - 1.0
        scores[~finite_mask] = floor
    return scores


def rank_of_target(scores: np.ndarray, target_idx: int) -> int:
    target_score = float(scores[target_idx])
    return int(np.sum(scores > target_score) + 1)


def evaluate_at_k_direct(sim_matrix: np.ndarray, id_lookup: dict[int, int], directed_eval_pairs, k: int = 10) -> dict[str, float]:
    hits = 0
    ndcg = 0.0
    recall = 0.0
    map_score = 0.0
    used_pairs = 0

    for source_id, target_id in directed_eval_pairs:
        if source_id not in id_lookup or target_id not in id_lookup:
            continue
        source_idx = id_lookup[source_id]
        target_idx = id_lookup[target_id]
        scores = safe_row_scores(sim_matrix, source_idx)
        rank = rank_of_target(scores, target_idx)
        used_pairs += 1

        if rank <= k:
            hits += 1
            recall += 1.0
            ndcg += 1.0 / math.log2(rank + 1)
            map_score += 1.0 / rank

    if used_pairs == 0:
        raise ValueError("No usable evaluation pairs were found for the requested metric.")

    return {
        "Hit@10": hits / used_pairs,
        "NDCG@10": ndcg / used_pairs,
        "Recall@10": recall / used_pairs,
        "MAP@10": map_score / used_pairs,
        "n_eval_pairs": used_pairs,
    }


def rank_fusion_scores_for_source(source_id: int, sim_bge_matrix: np.ndarray, sim_gcn_matrix: np.ndarray) -> np.ndarray:
    sims_bge = sim_bge_matrix[full_id_to_idx[source_id], bge_common_indices_rrf]
    sims_gcn = sim_gcn_matrix[gcn_id_to_idx[source_id], gcn_common_indices_rrf]

    ranked_bge = np.argsort(-sims_bge)
    ranked_gcn = np.argsort(-sims_gcn)
    rankpos_bge = np.empty(len(common_ids_rrf), dtype=np.int32)
    rankpos_gcn = np.empty(len(common_ids_rrf), dtype=np.int32)
    rankpos_bge[ranked_bge] = np.arange(1, len(common_ids_rrf) + 1, dtype=np.int32)
    rankpos_gcn[ranked_gcn] = np.arange(1, len(common_ids_rrf) + 1, dtype=np.int32)
    return (1.0 / (K_RRF + rankpos_bge)) + (1.0 / (K_RRF + rankpos_gcn))


def evaluate_at_k_rrf(directed_eval_pairs, k: int = 10) -> dict[str, float]:
    hits = 0
    ndcg = 0.0
    recall = 0.0
    map_score = 0.0
    used_pairs = 0
    score_cache: dict[int, np.ndarray] = {}

    for source_id, target_id in directed_eval_pairs:
        if source_id not in full_id_to_idx or source_id not in gcn_id_to_idx or target_id not in common_pos_rrf:
            continue
        if source_id not in score_cache:
            score_cache[source_id] = rank_fusion_scores_for_source(source_id, sim_bge, sim_gcn)
        scores = score_cache[source_id]
        target_pos = common_pos_rrf[target_id]
        target_score = float(scores[target_pos])
        rank = int(np.sum(scores > target_score) + 1)
        used_pairs += 1

        if rank <= k:
            hits += 1
            recall += 1.0
            ndcg += 1.0 / math.log2(rank + 1)
            map_score += 1.0 / rank

    if used_pairs == 0:
        raise ValueError("No usable evaluation pairs were found for the rank-fusion metric.")

    return {
        "Hit@10": hits / used_pairs,
        "NDCG@10": ndcg / used_pairs,
        "Recall@10": recall / used_pairs,
        "MAP@10": map_score / used_pairs,
        "n_eval_pairs": used_pairs,
    }


## Step 4 - AUC and PR-AUC helpers

We evaluate link discrimination on the same revised undirected branch by:

- using held-out undirected test edges as positives
- sampling the same number of undirected non-edges as negatives
- scoring each edge with the relevant similarity matrix

For rank fusion, the underlying score is directional, so we symmetrize an undirected edge
by averaging the `u -> v` and `v -> u` RRF scores.


In [5]:
def sample_negative_edges_undirected(graph_obj: nx.Graph, num_samples: int, seed: int = 42) -> list[tuple[int, int]]:
    rng = np.random.default_rng(seed)
    node_ids = np.array(sorted(int(node) for node in graph_obj.nodes()), dtype=np.int64)
    positive_edges = {tuple(sorted((int(u), int(v)))) for u, v in graph_obj.edges() if u != v}
    negatives: set[tuple[int, int]] = set()

    while len(negatives) < num_samples:
        u = int(rng.choice(node_ids))
        v = int(rng.choice(node_ids))
        if u == v:
            continue
        edge = (u, v) if u < v else (v, u)
        if edge in positive_edges or edge in negatives:
            continue
        negatives.add(edge)

    return sorted(negatives)


negative_edges = sample_negative_edges_undirected(graph_undirected, len(E_test), seed=SEED)
print("Negative undirected edges:", len(negative_edges))


def scores_for_edges_direct(sim_matrix: np.ndarray, id_lookup: dict[int, int], edges) -> np.ndarray:
    out = []
    for u, v in edges:
        if u not in id_lookup or v not in id_lookup:
            continue
        out.append(float(sim_matrix[id_lookup[u], id_lookup[v]]))

    scores = np.asarray(out, dtype=np.float32)
    finite_mask = np.isfinite(scores)
    if finite_mask.any() and (~finite_mask).any():
        floor = float(np.min(scores[finite_mask])) - 1.0
        scores[~finite_mask] = floor
    return scores


def scores_for_edges_rrf(edges) -> np.ndarray:
    score_cache: dict[int, np.ndarray] = {}
    out = []
    for u, v in edges:
        if (
            u not in full_id_to_idx
            or v not in full_id_to_idx
            or u not in gcn_id_to_idx
            or v not in gcn_id_to_idx
            or u not in common_pos_rrf
            or v not in common_pos_rrf
        ):
            continue
        if u not in score_cache:
            score_cache[u] = rank_fusion_scores_for_source(u, sim_bge, sim_gcn)
        if v not in score_cache:
            score_cache[v] = rank_fusion_scores_for_source(v, sim_bge, sim_gcn)
        score_uv = float(score_cache[u][common_pos_rrf[v]])
        score_vu = float(score_cache[v][common_pos_rrf[u]])
        out.append((score_uv + score_vu) / 2.0)
    return np.asarray(out, dtype=np.float32)


def compute_auc_metrics_from_scores(pos_scores: np.ndarray, neg_scores: np.ndarray) -> dict[str, float]:
    labels = np.concatenate([np.ones(len(pos_scores), dtype=np.int8), np.zeros(len(neg_scores), dtype=np.int8)])
    scores = np.concatenate([pos_scores, neg_scores])
    return {
        "ROC-AUC": float(roc_auc_score(labels, scores)),
        "PR-AUC": float(average_precision_score(labels, scores)),
        "n_positive_edges": int(len(pos_scores)),
        "n_negative_edges": int(len(neg_scores)),
    }


Negative undirected edges: 43223


## Step 5 - Evaluate all 10 models


In [6]:
cutoff_rows = []
auc_rows = []

for model in direct_models:
    cutoff_metrics = evaluate_at_k_direct(model["sim"], model["id_to_idx"], eval_pairs, k=TOP_K)
    pos_scores = scores_for_edges_direct(model["sim"], model["id_to_idx"], E_test)
    neg_scores = scores_for_edges_direct(model["sim"], model["id_to_idx"], negative_edges)
    auc_metrics = compute_auc_metrics_from_scores(pos_scores, neg_scores)
    cutoff_rows.append({"Model": model["name"], **cutoff_metrics})
    auc_rows.append({"Model": model["name"], **auc_metrics})

rrf_cutoff_metrics = evaluate_at_k_rrf(eval_pairs, k=TOP_K)
rrf_pos_scores = scores_for_edges_rrf(E_test)
rrf_neg_scores = scores_for_edges_rrf(negative_edges)
rrf_auc_metrics = compute_auc_metrics_from_scores(rrf_pos_scores, rrf_neg_scores)
cutoff_rows.append({"Model": "7. GCN + BGE-M3 (rank fusion)", **rrf_cutoff_metrics})
auc_rows.append({"Model": "7. GCN + BGE-M3 (rank fusion)", **rrf_auc_metrics})

df_cutoff10 = pd.DataFrame(cutoff_rows).sort_values("Hit@10", ascending=False).reset_index(drop=True)
df_auc = pd.DataFrame(auc_rows).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
df_eval31 = df_cutoff10.merge(df_auc, on="Model")

df_eval3_combined = pd.read_csv(EVAL3_CACHE_DIR / "combined_metrics.csv")
df_eval31_augmented = df_eval3_combined.merge(df_eval31, on="Model")

print("Computed eval-3.1 extension metrics for", len(df_eval31), "models.")


Computed eval-3.1 extension metrics for 10 models.


## Step 6 - Save eval-3.1 artifacts


In [7]:
df_cutoff10.to_csv(CACHE_DIR / "cutoff10_metrics.csv", index=False)
df_auc.to_csv(CACHE_DIR / "auc_metrics.csv", index=False)
df_eval31.to_csv(CACHE_DIR / "eval31_extension_metrics.csv", index=False)
df_eval31_augmented.to_csv(CACHE_DIR / "eval31_augmented_metrics.csv", index=False)

with open(CACHE_DIR / "negative_edges.pkl", "wb") as f:
    pickle.dump(negative_edges, f)

run_metadata = {
    "seed": SEED,
    "top_k": TOP_K,
    "n_models": int(len(df_eval31)),
    "n_positive_edges": len(E_test),
    "n_negative_edges": len(negative_edges),
    "n_directed_eval_pairs_for_cutoff_metrics": len(eval_pairs),
    "source_eval_cache": str(EVAL3_CACHE_DIR),
    "source_split": str(SPLIT_PATH),
    "rank_fusion_edge_score": "mean of directional RRF scores",
}
with open(CACHE_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)

print("Saved:")
print("-", CACHE_DIR / "cutoff10_metrics.csv")
print("-", CACHE_DIR / "auc_metrics.csv")
print("-", CACHE_DIR / "eval31_extension_metrics.csv")
print("-", CACHE_DIR / "eval31_augmented_metrics.csv")
print("-", CACHE_DIR / "negative_edges.pkl")
print("-", CACHE_DIR / "run_metadata.json")


Saved:
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.1\cutoff10_metrics.csv
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.1\auc_metrics.csv
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.1\eval31_extension_metrics.csv
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.1\eval31_augmented_metrics.csv
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.1\negative_edges.pkl
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.1\run_metadata.json


## Step 7 - Preview tables


In [8]:
print("Cutoff-10 metrics:")
display(df_cutoff10)

print("AUC / PR-AUC metrics:")
display(df_auc)

print("Eval-3 augmented with eval-3.1 extension metrics:")
display(df_eval31_augmented)


Cutoff-10 metrics:


,Model,Hit@10,NDCG@10,Recall@10,MAP@10,n_eval_pairs
0,10. BGE pooled + Node2Vec (corrected),0.086736,0.040913,0.086736,0.027275,86446
1,5. BGE-M3 + Node2Vec (corrected),0.086690,0.041304,0.086690,0.027778,86446
2,9. BGE pooled + Node2Vec (uncorrected),0.078349,0.035593,0.078349,0.022913,86446
3,4. BGE-M3 + Node2Vec (uncorrected),0.078338,0.035608,0.078338,0.022934,86446
4,2. Node2Vec (uncorrected),0.078130,0.035508,0.078130,0.022870,86446
5,3. Node2Vec (corrected),0.078095,0.035464,0.078095,0.022823,86446
6,1. BGE-M3 Only,0.076383,0.040601,0.076383,0.029794,86446
7,7. GCN + BGE-M3 (rank fusion),0.076325,0.035544,0.076325,0.023416,86446
8,6. GCN Only,0.034588,0.016149,0.034588,0.010660,86446
9,8. GraphSAGE Only,0.012227,0.005961,0.012227,0.004096,86446


AUC / PR-AUC metrics:


,Model,ROC-AUC,PR-AUC,n_positive_edges,n_negative_edges
0,2. Node2Vec (uncorrected),0.971666,0.971390,43223,43223
1,3. Node2Vec (corrected),0.971666,0.971376,43223,43223
2,4. BGE-M3 + Node2Vec (uncorrected),0.971005,0.971594,43223,43223
3,9. BGE pooled + Node2Vec (uncorrected),0.970989,0.971573,43223,43223
4,5. BGE-M3 + Node2Vec (corrected),0.958185,0.957947,43223,43223
5,10. BGE pooled + Node2Vec (corrected),0.956345,0.955885,43223,43223
6,7. GCN + BGE-M3 (rank fusion),0.935649,0.936250,43223,43223
7,6. GCN Only,0.895377,0.894558,43223,43223
8,1. BGE-M3 Only,0.885149,0.888556,43223,43223
9,8. GraphSAGE Only,0.869078,0.856113,43223,43223


Eval-3 augmented with eval-3.1 extension metrics:


,Model,Hit@1,Hit@10_x,Hit@50,Hit@100,MRR,NDCG,Recall,MAP,n_eval_pairs_x,...,top_k,Hit@10_y,NDCG@10,Recall@10,MAP@10,n_eval_pairs_y,ROC-AUC,PR-AUC,n_positive_edges,n_negative_edges
0,10. BGE pooled + Node2Vec (corrected),0.010284,0.086736,0.314566,0.478090,0.040967,0.116046,0.478090,0.039236,86446,...,20,0.086736,0.040913,0.086736,0.027275,86446,0.956345,0.955885,43223,43223
1,5. BGE-M3 + Node2Vec (corrected),0.010712,0.086690,0.324064,0.498126,0.042058,0.120153,0.498126,0.040279,86446,...,20,0.086690,0.041304,0.086690,0.027778,86446,0.958185,0.957947,43223,43223
2,9. BGE pooled + Node2Vec (uncorrected),0.007565,0.078349,0.326412,0.507415,0.037610,0.117589,0.507415,0.035782,86446,...,20,0.078349,0.035593,0.078349,0.022913,86446,0.970989,0.971573,43223,43223
3,4. BGE-M3 + Node2Vec (uncorrected),0.007577,0.078338,0.326400,0.507392,0.037634,0.117605,0.507392,0.035805,86446,...,20,0.078338,0.035608,0.078338,0.022934,86446,0.971005,0.971594,43223,43223
4,2. Node2Vec (uncorrected),0.007542,0.078118,0.325244,0.505981,0.037487,0.117224,0.505981,0.035655,86446,...,20,0.078130,0.035508,0.078130,0.022870,86446,0.971666,0.971390,43223,43223
5,3. Node2Vec (corrected),0.007519,0.078083,0.325244,0.505981,0.037469,0.117209,0.505981,0.035637,86446,...,20,0.078095,0.035464,0.078095,0.022823,86446,0.971666,0.971376,43223,43223
6,7. GCN + BGE-M3 (rank fusion),0.008479,0.076325,0.231381,0.337737,0.033468,0.086187,0.337737,0.031694,86446,...,20,0.076325,0.035544,0.076325,0.023416,86446,0.935649,0.936250,43223,43223
7,1. BGE-M3 Only,0.013534,0.075689,0.196921,0.277260,0.036616,0.078753,0.277260,0.035146,86446,...,20,0.076383,0.040601,0.076383,0.029794,86446,0.885149,0.888556,43223,43223
8,6. GCN Only,0.003841,0.034588,0.141186,0.229103,0.018029,0.052966,0.229103,0.016304,86446,...,20,0.034588,0.016149,0.034588,0.010660,86446,0.895377,0.894558,43223,43223
9,8. GraphSAGE Only,0.001747,0.012227,0.049603,0.087153,0.007569,0.019899,0.087153,0.006138,86446,...,20,0.012227,0.005961,0.012227,0.004096,86446,0.869078,0.856113,43223,43223
